In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

import pandas as pd
import json
import glob
import os
import numpy as np


base_dir = "/content/drive/MyDrive/Data/outputs_stage0_4_full450_cuda5090"
cases_dir = os.path.join(base_dir, "cases")

# 检查路径是否存在
if os.path.exists(base_dir):
    print(f"✅ 成功找到数据目录: {base_dir}")
    print("包含的文件:", os.listdir(base_dir))
else:
    print(f"❌ 找不到目录，请检查挂载状态或路径是否正确。")

✅ 成功找到数据目录: /content/drive/MyDrive/Data/outputs_stage0_4_full450_cuda5090
包含的文件: ['ctrate_case_summary.csv', 'radgenome_case_summary.csv', 'summary.csv', 'cache', 'cases']


In [6]:
print("1. 数据集全局统计 (包含违规率)")

# 读取汇总 CSV
summary_path = os.path.join(base_dir, "summary.csv")
df_summary = pd.read_csv(summary_path)

# 计算每个病例的“平均每句违规次数”
df_summary['violation_rate'] = df_summary['n_violations'] / df_summary['n_sentences']

# 按数据集 (ctrate / radgenome) 分组统计
stats = df_summary.groupby("dataset", as_index=False).agg(
    病例总数=("case_id", "count"),
    平均Token数=("n_tokens", "mean"),
    平均句子数=("n_sentences", "mean"),
    规则违规总数=("n_violations", "sum"),
    平均每句违规率=("violation_rate", "mean")  # <--- 把你的需求加在这里
)

# 格式化一下小数位，让表格更好看
stats['平均Token数'] = stats['平均Token数'].round(1)
stats['平均句子数'] = stats['平均句子数'].round(1)
stats['平均每句违规率'] = stats['平均每句违规率'].round(4)

display(stats)

========== 1. 数据集全局统计 (包含违规率) ==========


,dataset,病例总数,平均Token数,平均句子数,规则违规总数,平均每句违规率
0,ctrate,450,64.0,3.9,1551,0.8944
1,radgenome,450,64.0,8.0,2780,0.7722


In [7]:
print("2. R1-R5 规则触发分布")

rule_count = {}
total_sentences = 0
violating_sentences = 0

# 匹配所有 trace.jsonl 文件 (格式: cases/数据集/case_id/trace.jsonl)
trace_files = glob.glob(os.path.join(cases_dir, "*", "*", "trace.jsonl"))
print(f"找到 {len(trace_files)} 个 trace.jsonl 文件，正在分析...")

for p in trace_files:
    with open(p, "r", encoding="utf-8") as f:
        # 逐行读取 JSON
        lines = [json.loads(x) for x in f if x.strip()]

    # 跳过第一行 case_meta，直接看后续的句子路由记录
    for row in lines[1:]:
        total_sentences += 1
        violations = row.get("violations", [])
        if violations:
            violating_sentences += 1
            for v in violations:
                rid = v.get("rule_id", "UNKNOWN")
                rule_count[rid] = rule_count.get(rid, 0) + 1

# 转换为易读的表格
df_rules = pd.DataFrame(list(rule_count.items()), columns=['规则 ID (Rule_ID)', '触发次数 (Count)'])
df_rules = df_rules.sort_values('触发次数 (Count)', ascending=False).reset_index(drop=True)

display(df_rules)
print(f"\n总计分析了 {total_sentences} 个句子，其中 {violating_sentences} 个句子发生了至少一次违规。")

2. R1-R5 规则触发分布
找到 882 个 trace.jsonl 文件，正在分析...


,规则 ID (Rule_ID),触发次数 (Count)
0,R2_ANATOMY,2151
1,R1_LATERALITY,1396
2,R4_SIZE,733



总计分析了 5282 个句子，其中 3486 个句子发生了至少一次违规。


In [8]:
import os
import json
import random

# 确保 cases_dir 已经正确定义（基于你前面的代码）
# cases_dir = "/content/drive/MyDrive/Data/outputs_stage0_4_full450_cuda5090/cases"

def inspect_case_trace(dataset_name, case_id):
    trace_path = os.path.join(cases_dir, dataset_name, case_id, "trace.jsonl")
    if not os.path.exists(trace_path):
        print(f"❌ 找不到该病例的追踪文件: {trace_path}")
        return

    with open(trace_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f]

    meta = data[0] # 第一行是 case_meta
    print(f"========== 抽查病例: {case_id} ({dataset_name}) ==========")
    print(f"参数配置: B={meta.get('B')}, 检索k={meta.get('k')}, 空间先验 lambda={meta.get('lambda_spatial')}")
    print("-" * 50)

    # 打印每句话路由情况
    for s in data[1:]:
        status = "❌" if s.get('violations') else "✅"
        print(f"{status} 第 {s.get('sentence_index')} 句: {s.get('sentence_text')}")
        print(f"   引用的 Tokens ID: {s.get('topk_token_ids')}")

        if s.get('violations'):
            for v in s['violations']:
                print(f"   -> [违规警告] 规则: {v.get('rule_id')}, 严重度: {v.get('severity')}, 信息: {v.get('message')}")
        print()

def random_sample_and_inspect(dataset_name, num_samples=3):
    """从指定数据集中随机抽查 num_samples 个病例"""
    dataset_path = os.path.join(cases_dir, dataset_name)

    # 获取该数据集下所有的 case 文件夹名称
    if not os.path.exists(dataset_path):
        print(f"❌ 找不到数据集目录: {dataset_path}")
        return

    all_cases = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

    if not all_cases:
        print(f"❌ 数据集 {dataset_name} 下没有找到病例。")
        return

    # 随机抽取指定数量的病例（如果总数不足则全部抽取）
    sampled_cases = random.sample(all_cases, min(num_samples, len(all_cases)))

    print(f"\n\n{'#'*60}")
    print(f" 开始随机抽查 {dataset_name} 数据集 (共 {len(sampled_cases)} 个)")
    print(f"{'#'*60}\n")

    for case_id in sampled_cases:
        inspect_case_trace(dataset_name, case_id)

# ================= 运行抽查 =================
# 分别对两个数据集进行抽查，各 3 个
random_sample_and_inspect('ctrate', num_samples=3)
random_sample_and_inspect('radgenome', num_samples=3)



############################################################
 开始随机抽查 ctrate 数据集 (共 3 个)
############################################################

========== 抽查病例: train_11111 (ctrate) ==========
参数配置: B=64, 检索k=8, 空间先验 lambda=0.3
--------------------------------------------------
✅ 第 0 句: Cardiomegaly
   引用的 Tokens ID: [62, 30, 46, 57, 14, 37, 24, 56]

❌ 第 1 句: Aneurysmatic dilatation of the ascending aorta-arch aorta,
   引用的 Tokens ID: [52, 59, 54, 57, 58, 53, 2, 8]
   -> [违规警告] 规则: R1_LATERALITY, 严重度: 1.0, 信息: Laterality mismatch with claim=right.

❌ 第 2 句: Pleural effusion on the left
   引用的 Tokens ID: [8, 24, 40, 56, 6, 10, 9, 20]
   -> [违规警告] 规则: R1_LATERALITY, 严重度: 1.0, 信息: Laterality mismatch with claim=left.

❌ 第 3 句: Area of consolidation-atelectasis with air bronchogram in the lower lobe of the left lung
   引用的 Tokens ID: [30, 46, 45, 8, 11, 13, 14, 59]
   -> [违规警告] 规则: R1_LATERALITY, 严重度: 1.0, 信息: Laterality mismatch with claim=left.
   -> [违规警告] 规则: R2_ANATOMY, 严重度: 0